# Ni powder diffraction: integration and Rietveld refinement

This notebook converts `Ni.tiff` to a one-dimensional powder profile with pyFAI through `easyXRD`, applies the supplied detector mask and geometry, and refines the cubic Ni phase in GSAS-II using the CIF structure and instrument parameters stored in the supplied GPX file.

The workflow follows the `easyXRD` example notebooks, especially their `load_xrd_data`, `load_phases`, `setup_gsas2_refiner`, and refinement APIs ([easyXRD examples](https://github.com/MehmetTopsakal/easyXRD_examples)). Run from this directory with the user's Pixi Python. The environment may need `NUMBA_DISABLE_JIT=1` for the packaged `pybaselines` cache to work in a read-only site-packages install; this does not affect pyFAI integration or GSAS-II refinement.

In [1]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import easyxrd
from easyxrd.core import exrd

ROOT = Path.cwd()
IMAGE = ROOT / "Ni.tiff"
PONI = ROOT / "_calibration.poni"
MASK = ROOT / "_mask.edf"
CIF = ROOT / "Ni.cif"
INSTRUMENT_GPX = ROOT / "_instrument_parameters.gpx"
OUTPUT = ROOT

# easyXRD needs a writable scratch area to stage the GSAS-II project.
SCRATCH = ROOT / ".easyxrd_scratch"
SCRATCH.mkdir(exist_ok=True)
easyxrd.set_defaults("easyxrd_scratch_path", str(SCRATCH))

for path in [IMAGE, PONI, MASK, CIF, INSTRUMENT_GPX]:
    assert path.is_file(), f"Missing required input: {path.name}"

print("easyXRD version:", easyxrd.version("easyxrd"))
print("Scratch directory:", SCRATCH)



Imported easyxrd with the following configuration:

easyxrd_scratch_path : /home/mt/.easyxrd_scratch
gsasii_lib_path : /users/software/mpixi/.pixi/envs/default/lib/python3.14/site-packages/GSASII
mp_api_key : dHgNQRNYS..........
easyXRD version: 2026.9.20
Scratch directory: /home/mt/repos/easyXRD_with_agents/00-Ni/GPT-6-Luna-Medium/.easyxrd_scratch


## 1. Integrate the detector image to 1D

The PONI file supplies the detector geometry and wavelength (0.1799 Å). The EDF mask is passed to pyFAI so masked detector pixels do not contribute to the integration. Integration is done directly to 1D in scattering-vector units over q = 1–8 Å⁻¹ at 0.002 Å⁻¹ spacing; the wavelength is retained in the dataset so the profile can be expressed on a 2θ axis for GSAS-II.

In [2]:
analysis = exrd()
analysis.load_xrd_data(
    integrate2d=False,
    from_tiff_file=str(IMAGE),
    poni_file=str(PONI),
    mask_file=str(MASK),
    radial_range=[1.0, 8.0],
    delta_q=0.002,
    plot=False,
)

profile = analysis.ds.i1d
q = profile.radial.values.astype(float)
intensity = profile.values.astype(float)
wavelength = float(profile.attrs["wavelength_in_angst"])
two_theta = np.rad2deg(2 * np.arcsin(q * wavelength / (4 * np.pi)))

# Save a conventional 2-column 2theta/intensity profile and a richer CSV.
analysis.export_i1d_to(to=str(OUTPUT / "Ni_integrated.xy"), mode="xy")
pd.DataFrame({"q_A^-1": q, "two_theta_deg": two_theta, "intensity": intensity}).to_csv(
    OUTPUT / "Ni_integrated.csv", index=False
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(two_theta, intensity, lw=0.7)
ax.set(xlabel=r"2$\theta$ (degrees)", ylabel="Intensity (a.u.)", title="Ni 1D profile after pyFAI integration")
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT / "Ni_integrated.png", dpi=180)
from IPython.display import display
display(fig)
plt.close(fig)
print(f"Integrated {len(q):,} bins; wavelength = {wavelength:.5f} Å")
print("Saved Ni_integrated.xy and Ni_integrated.csv")

Integrated 3,500 bins; wavelength = 0.17990 Å
Saved Ni_integrated.xy and Ni_integrated.csv


<Figure size 1000x400 with 1 Axes>

![Integrated Ni diffraction profile](Ni_integrated.png)

## 2. Load the Ni phase and instrument parameters

The CIF provides the starting crystal structure (Fm-3m, cubic `a = 3.525576 Å`). `setup_gsas2_refiner` imports the instrument profile terms from `_instrument_parameters.gpx` and initially performs the package's Le Bail setup refinement. The Le Bail flag is then turned off before the structural Rietveld refinement.

In [3]:
analysis.load_phases(
    from_phases_dict=[{"label": "Ni", "cif": str(CIF)}],
    plot=False,
)
analysis.setup_gsas2_refiner(
    instprm_from_gpx=str(INSTRUMENT_GPX),
    do_1st_refinement=True,
    plot=False,
)
print("Phase(s):", list(analysis.phases))
print("Initial Le Bail fit:", f"Rwp = {analysis.ds.attrs['Rwp']:.3f}%", f"GoF = {analysis.ds.attrs['GOF']:.3f}")


 ⏩--1st refinement with LeBail is completed. Rwp/GoF is 26.547/1.316 

Phase(s): ['Ni']
Initial Le Bail fit: Rwp = 26.547% GoF = 1.316


## 3. Rietveld refinement

After the Le Bail setup, turn off Le Bail extraction and refine a six-term Chebyshev background, the GSAS-II U/V/W and zero instrument terms, and the cubic Ni unit-cell parameter. All phase and histogram settings are staged in the GSAS-II project through easyXRD; the final refinement updates the plotted calculated profile and fit statistics.

In [4]:
analysis.set_LeBail(to=False, refine=False)
analysis.set_background_refinement(set_num_coeffs_to=6)
analysis.set_instrument_parameters_refinement(
    set_inst_pars_to_refine=["U", "V", "W", "Zero"]
)
analysis.set_cell_parameters_refinement()
rietveld_summary = analysis.refine(plot=False)
analysis.gpx_saver()

# Keep a convenient, named copy of the final GSAS-II project in this folder.
gpx_source = Path(analysis.gsasii_run_directory) / "gsas.gpx"
shutil.copy2(gpx_source, OUTPUT / "Ni_Rietveld.gpx")

metrics = {
    "Rwp_percent": float(analysis.ds.attrs["Rwp"]),
    "GoF": float(analysis.ds.attrs["GOF"]),
    "chi_squared": float(analysis.ds.attrs["chisq"]),
    "converged": analysis.ds.attrs.get("converged"),
    "aborted_flag": analysis.ds.attrs.get("Aborted"),
    "Ni_a_A": float(analysis.ds.attrs["PhaseInd_0_cell_a"]),
    "Ni_a_start_A": float(analysis.ds.attrs["PhaseInd_0_cell_a_previous"]),
}
print(rietveld_summary)
print(pd.Series(metrics).to_string())
print("Saved Ni_Rietveld.gpx")

Rwp/GoF is now 14.622/0.726 (was 26.547(-44.92%)/1.316(-44.83%✨))
Rwp_percent       14.622458
GoF                0.726032
chi_squared     1838.604492
converged              True
aborted_flag           True
Ni_a_A             3.522891
Ni_a_start_A       3.525576
Saved Ni_Rietveld.gpx


## 4. Inspect the fitted pattern

The top panel overlays observed intensity, the final GSAS-II calculated profile, and the fitted background. The lower panel shows observed-minus-calculated residuals. The output plot and profile files are saved beside this notebook for reuse.

In [5]:
observed = analysis.ds.i1d.values
calculated = analysis.ds.i1d_refined.values
background = analysis.ds.i1d_gsas_background.values
residual = observed - calculated

fig, (ax, ax_diff) = plt.subplots(
    2, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]}
)
ax.plot(two_theta, observed, color="black", lw=0.65, label="Observed")
ax.plot(two_theta, calculated, color="tab:red", lw=0.8, label="Rietveld calculated")
ax.plot(two_theta, background, color="tab:blue", lw=0.8, label="Background")
ax.set_ylabel("Intensity (a.u.)")
ax.legend(frameon=False, ncol=3)
ax.grid(alpha=0.18)
ax_diff.axhline(0, color="0.4", lw=0.7)
ax_diff.plot(two_theta, residual, color="0.25", lw=0.6)
ax_diff.set(xlabel=r"2$\theta$ (degrees)", ylabel="Obs. − calc.")
ax_diff.grid(alpha=0.18)
fig.suptitle(f"Ni Rietveld fit: Rwp = {metrics['Rwp_percent']:.2f}%, GoF = {metrics['GoF']:.3f}")
fig.tight_layout()
fig.savefig(OUTPUT / "Ni_Rietveld_fit.png", dpi=180)
from IPython.display import display
display(fig)
plt.close(fig)

<Figure size 1000x600 with 2 Axes>

![Ni Rietveld fit and residual](Ni_Rietveld_fit.png)

## Results and notes

The final fit reports Rwp and GoF from the GSAS-II covariance results and a refined cubic Ni lattice constant. The unit-cell refinement is close to the supplied CIF starting value. GSAS-II's `converged` and `Aborted` status flags are both included in the output above because the library can report its cycle-stop state separately from the convergence flag. Review the residual plot and fit range when deciding whether additional terms (for example size/strain broadening or preferred orientation) are justified by the data.

Generated files: `Ni_integrated.xy`, `Ni_integrated.csv`, `Ni_integrated.png`, `Ni_Rietveld.gpx`, and `Ni_Rietveld_fit.png`.